**[🏠 Course Home](../README.md)** | **📖 [Conceptual Track](../conceptual/12_bayesian_regression_and_glms.ipynb)** | **🤖 [AI Track](12_bayesian_regression_and_glms.ipynb)** | **🐍 [Python Track](../python/12_bayesian_regression_and_glms.ipynb)** | **🔵 [R Track](../r/12_bayesian_regression_and_glms.ipynb)** | **⬅️ [Prev: Ch 11: Two-Group t-Test](11_t_test_frequentist_vs_bayesian.ipynb)** | **➡️ [Next: Ch 13: Hierarchical Models & Shrinkage](13_hierarchical_models_and_partial_pooling.ipynb)**

*🧭 Part III: Applied Modeling & Hierarchy — Chapter 12 of 22*

---

# Chapter 12: Bayesian Regression & GLMs
### *The Two Uncertainty Ribbons, Centered Geometry, and Aleatoric Guarantees*

---

> **🧭 Course Context & Why This Module Matters in the AI Era:**
>
> In Chapters 1 through 11, we estimated static, isolated parameters: a
> single conversion rate, a fleet-wide sensor location, or the difference
> between two treatment groups. In production systems, variables are rarely
> isolated. Latency depends conditionally on traffic volume (QPS), memory
> utilization scales with concurrent connections, and component failure
> rates accelerate with operating temperature.
>
> When modern engineers ask Large Language Models (LLMs) to construct
> predictive models or certify service-level agreements (SLAs), AI assistants
> suffer from three catastrophic failure modes:
> 1. **The Single-Ribbon Fallacy**: Conflating the uncertainty of the average
>    trend line $\mu(x)$ with the dispersion of individual future requests
>    $y_{\text{new}}$. This mistake leads teams to publish SLAs that fail on
>    over $25\%$ of production requests.
> 2. **Uncentered Geometric Pathology**: Failing to center explanatory variables
>    ($x - \bar{x}$), producing severe covariance ridges between the intercept
>    and slope that induce silent Hamiltonian Monte Carlo (HMC) sampling
>    breakdowns.
> 3. **Linear Link Boundary Violations**: Fitting unconstrained linear
>    equations to bounded probabilities, predicting absurd negative failure
>    rates ($p < 0$) or probabilities exceeding $1.0$.
>
> This module teaches you how to prompt and constrain AI coding agents as
> rigorous probabilistic systems engineers.


## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

### The Prompt That Fools the AI

An infrastructure reliability engineer wants to predict API service latency as
a function of incoming request volume (QPS) to establish a contractual 95%
Service Level Agreement (SLA). They also want to predict the probability of
server memory exhaustion based on CPU temperature.

They submit the following conversational prompt to an advanced AI coding
assistant:

```text
Hey! We have telemetry data showing server latency (y in ms) as a function of
QPS load (x). Fit a Bayesian linear regression in Python to find the 95%
confidence band for SLA provisioning. Also, fit a linear regression on server
temperature to predict whether a server crashes (1) or stays healthy (0).
Provide clean visualization code and explain our 95% latency guarantee at 800
QPS.
```

---

### The Plausible AI Hallucination

The AI enthusiastically generates Python code using PyMC or Stan. The code
defines:

```python
# AI Generated Code (Flawed!)
with pm.Model() as model:
    alpha = pm.Normal("alpha", mu=0, sigma=100)
    beta = pm.Normal("beta", mu=0, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=20)
    
    mu = alpha + beta * qps_data  # Uncentered!
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=latency_data)
    trace = pm.sample(2000)

# Computing the "95% SLA Band"
qps_grid = np.linspace(100, 1000, 100)
mu_samples = trace.posterior["alpha"].values + trace.posterior["beta"].values * qps_grid[:, None]
sla_lower = np.percentile(mu_samples, 2.5, axis=1)
sla_upper = np.percentile(mu_samples, 97.5, axis=1)
```

The AI provides the following commentary:
> *"At 800 QPS, your server latency is guaranteed with 95% confidence to fall
> between **118.2 ms** and **124.6 ms**. You can safely set your customer SLA
> timeout to **125 ms**! For server crash prediction, our linear regression
> shows that at 45°C, the estimated crash probability is **-0.08** (essentially
> zero)."*

---

### Forensic Post-Mortem: Why the System Crashes

```
     NAIVE AI SLA HORROR: CONFLATING THE TWO UNCERTAINTY RIBBONS
 Latency
  (ms)
   160 |                                          *   *  *  True Outliers
   150 |---------------------------------------*------------- [Real 95% Bound: 150ms]
   140 |                             *  *   *       *  *
   130 |                         *        *    *  *
   125 |===================================================== [AI Claimed SLA: 125ms]
   120 |                     ------------------.--------      [AI Mean Ribbon:
   115 |                    /  Mean Line μ(x)  /               118ms - 124ms]
   100 |         *    *  * -------------------                (Width = 6ms!)
    80 |   *  *    *   *   *
       +---------------------------------------------------
      100           400           700           900  QPS
```

#### 1. Conflating Mean Credible Ribbon with Posterior Predictive Ribbon
The AI plotted the $95\%$ credible interval of the **mean parameter** $\mu(x) =
\mathbb{E}[y \mid x]$. This interval reflects purely **epistemic uncertainty**
(how well the regression line is estimated). Because $N = 100$, the sample
mean is known with tight precision ($\text{width} \approx 6.4\text{ ms}$).

However, an individual request does not arrive at the mean line! It arrives
with residual physical dispersion $\sigma = 15.0\text{ ms}$ (aleatoric
uncertainty):
$$y_{\text{new}} \sim \mathcal{N}\left(\mu(x),\, \sigma^2\right)$$
The variance of a new observation is:
$$\operatorname{Var}(y_{\text{new}} \mid x) = \operatorname{Var}(\mu(x)) + \sigma^2$$
The true $95\%$ **Posterior Predictive Interval** at $800$ QPS spans from
**$91.8\text{ ms}$ to $150.2\text{ ms}$** ($\text{width} \approx 58.4\text{
ms}$). Setting an SLA at $125\text{ms}$ causes **over $25\%$ of customer
requests to breach the contract**, triggering catastrophic outages and penalty
clauses!

#### 2. Uncentered Predictors & Correlated Valleys
By evaluating `mu = alpha + beta * qps`, the intercept $\alpha$ represents
latency at $0\text{ QPS}$ (an extrapolation outside the operational range).
Because the average QPS is $\bar{x} = 550$, tilting the slope $\beta$ forces
$\alpha$ to swing drastically in the opposite direction:
$$\operatorname{Cov}(\hat{\alpha}, \hat{\beta}) = -\bar{x} \operatorname{Var}(\hat{\beta}) \implies \rho(\alpha, \beta) \approx -0.98$$
This creates an ultra-narrow diagonal canyon in parameter space, causing HMC
trajectories to make tiny steps, inflate $\hat{R}$, and drop effective sample
size (ESS) by an order of magnitude.

#### 3. Linear Probability Impossibility
Predicting crash events with an unconstrained linear equation produces
negative probabilities ($p = -0.08$) and probabilities greater than $1.0$ at
high temperatures. A Generalized Linear Model (GLM) with a Logit link function
$\text{logit}(p) = \ln(p / (1-p))$ is required.


## Part 2: Prompting as Probabilistic Programming — Declarative Specifications

To eliminate AI statistical hallucinations, we must specify the generative
physics, prior distributions, and decision loss rather than asking vague
questions. Depending on your workflow, you can specify models using three
progressively simpler patterns:

---

### Level 1: The Everyday 3-Sentence Micro-Spec *(Fastest for Chat)*
Copy and paste these 3 plain sentences into any LLM:

```markdown
1. Data: Server latency vs QPS load; center the predictor x_c = x - x_bar
   to orthogonalize intercept and slope.
2. Dual Bands: Distinguish the narrow 95% Credible Band of the mean line mu(x)
   from the wide 95% Posterior Predictive Band y_new ~ Normal(mu(x), sigma).
3. GLM Link: For binary crash outcomes, use a logistic link logit(p) = alpha + beta*x
   to guarantee valid probabilities in [0, 1].
```

---

### Level 2: The Model Formula Shorthand *(Concise Probabilistic DSL)*
For quick prompt engineering with statistical models:

```text
Linear:   y_i ~ Normal(alpha + beta * (x_i - x_bar), sigma)
Logistic: y_i ~ Bernoulli(logit^{-1}(alpha + beta * T_i))
SLA Band: y_new ~ Normal(mu(x), sigma)  (Posterior Predictive Band)
```

---

### Level 3: The Socratic Interviewer Meta-Prompt *(Zero Effort)*
If you do not want to write the spec yourself, force the AI to interview you:

> *"Act as a Systems Regression Statistician. I am modeling server latency against
traffic load. Ask me 3 questions about predictor centering, mean vs individual
prediction bands, and SLA risk before fitting a regression."*

---

<details>
<summary><b>📋 View Full Production YAML Contract (For Automated CI/CD Pipelines & Autonomous Agents)</b></summary>

```yaml
declarative_probabilistic_specification:
  module: "bayesian_regression_and_glms"
  domain: "server_performance_and_hardware_reliability"
  
  observables:
    continuous_response:
      name: "latency_ms"
      symbol: "y"
      domain: "Real(> 0)"
    binary_response:
      name: "node_crash"
      symbol: "z"
      domain: "Binary {0, 1}"
    predictors:
      load:
        name: "qps"
        symbol: "x"
        centering: "x_c = x - mean(x)"
      thermal:
        name: "temp_celsius"
        symbol: "T"
        centering: "T_c = T - mean(T)"

  linear_regression_model:
    mean_structure: "mu_i = alpha_c + beta * (x_i - x_bar)"
    likelihood: "y_i ~ Normal(mu_i, sigma)"
    priors:
      alpha_c: "Normal(mu=120.0, sigma=20.0)  # Latency at mean QPS"
      beta: "Normal(mu=0.08, sigma=0.04)     # Latency increase per QPS"
      sigma: "Exponential(lam=0.067)          # Expected residual noise ~ 15ms"

  logistic_glm_model:
    link_function: "logit(p_i) = gamma_c + delta * (T_i - T_bar)"
    likelihood: "z_i ~ Bernoulli(p_i)"
    priors:
      gamma_c: "Normal(mu=-3.0, sigma=1.0)   # Baseline crash log-odds at mean temp"
      delta: "Normal(mu=0.15, sigma=0.05)    # Thermal sensitivity coefficient"

  predictive_ribbon_contracts:
    epistemic_band:
      name: "Credible Interval of Mean Line"
      target: "mu(x) = alpha_c + beta * (x - x_bar)"
      percentiles: [2.5, 97.5]
      purpose: "Evaluating scientific parameter knowledge"
    aleatoric_predictive_band:
      name: "Posterior Predictive Interval"
      target: "y_new ~ Normal(mu(x), sigma)"
      percentiles: [2.5, 97.5]
      purpose: "Contractual SLA sizing and infrastructure provisioning"

  decision_contract:
    sla_target_percentile: 0.95
    sla_confidence_level: 0.95
    safe_operating_temperature_rule:
      condition: "P(Crash | T_safe) <= 0.05 at 95% posterior confidence"
```

</details>


## Part 3: The Calibrated Agent Response & Telemetry

### Dataset Profile & MCMC Execution
* Sample size: $N = 100$ latency points across loads $x \in [100, 1000]$ QPS.
* Mean load: $\bar{x} = 550.0\text{ QPS}$.
* Sampler: No-U-Turn Sampler (NUTS), 4 chains, 2,000 warmup, 2,000 draws.
* Divergences: $0$. Max $\hat{R} = 1.0002$. Min ESS = $4,821$.

```text
================================================================================
                    MCMC POSTERIOR TELEMETRY LOG (CENTERED)
================================================================================
Parameter      Mean      SD   95% HPD Lower   95% HPD Upper    R-hat    ESS Bulk
--------------------------------------------------------------------------------
alpha_c      120.12    1.51          117.15          123.08   1.0001        5120
beta           0.081   0.005           0.071           0.091   1.0002        4892
sigma         14.88    1.08          12.82           17.04   1.0001        5410
--------------------------------------------------------------------------------
Cov(alpha_c, beta) = -0.0018  ==>  Corr(alpha_c, beta) = -0.024 (Orthogonal!)
================================================================================
```

---

### The Two Uncertainty Ribbons: Quantile Breakdown

| Metric at Load $x$ | 200 QPS ($x_c = -350$) | 550 QPS ($x_c = 0$) | 800 QPS ($x_c = +250$) |
| :--- | :--- | :--- | :--- |
| **Mean Line $\mu(x)$ [Epistemic]** | $91.8 \pm 2.2\text{ ms}$ | $120.1 \pm 1.5\text{ ms}$ | $140.4 \pm 2.0\text{ ms}$ |
| **95% Credible Band of $\mu(x)$** | $[87.5,\, 96.1]\text{ ms}$ | $[117.2,\, 123.1]\text{ ms}$ | $[136.5,\, 144.3]\text{ ms}$ |
| **Mean Line Band Width** | **$8.6\text{ ms}$** | **$5.9\text{ ms}$** | **$7.8\text{ ms}$** |
| **Posterior Predictive $y_{\text{new}}$** | $91.8 \pm 15.0\text{ ms}$ | $120.1 \pm 14.9\text{ ms}$ | $140.4 \pm 15.0\text{ ms}$ |
| **95% Predictive Band for Points**| **$[62.2,\, 121.4]\text{ ms}$** | **$[90.7,\, 149.5]\text{ ms}$** | **$[110.8,\, 170.0]\text{ ms}$** |
| **Predictive Band Width** | **$59.2\text{ ms}$** | **$58.8\text{ ms}$** | **$59.2\text{ ms}$** |

```
                       THE DUAL UNCERTAINTY BEAM
 Latency
  (ms)
   170 |                                             *  [Upper Pred Band: 170ms]
   160 |                                         *   *
   150 |                                 *   *       *
   140 |                             ===.====.========  [Mean Ribbon: 136-144ms]
   130 |                         *   *   *   *
   120 |                 ========.===.========          [Mean Ribbon: 117-123ms]
   110 |             *   *   *   *                      [Lower Pred Band: 111ms]
   100 |     ========.===.========                      [Mean Ribbon: 88-96ms]
    90 | *   *   *   *
    60 | * [Lower Pred Band: 62ms]
       +-------------------------------------------------------------
       100          300           550           800          1000 QPS
```

---

### GLM Failure Rate Analysis: Safe Operating Temperature Envelope
* Dataset: $N = 80$ server runs across temperatures $T \in [40, 90]^\circ\t
  across operational thermal bounds.
* Mean Temperature: $\bar{T} = 65.0^\circ\text{C}$.
* Logit Model: $\text{logit}(p) = \gamma_c + \delta (T - 65.0)$.
* Posterior Estimates: $\gamma_c = -2.85 \pm 0.32$, $\delta = 0.162 \pm 0.021$.

```text
================================================================================
               RELIABILITY ENVELOPE DECISION TELEMETRY
================================================================================
Policy: Failure probability must not exceed 5.0% with 95% posterior confidence.
Operating Target: Find T_safe such that P(p(T_safe) <= 0.05 | Data) >= 0.95.

Median Failure Curve:
  p_median(T) = logit^{-1}(-2.85 + 0.162 * (T - 65.0))
  Crossing at p = 0.05: T = 65.0 + (logit(0.05) - (-2.85)) / 0.162 = 64.4 deg C.

95th Percentile Upper Failure Curve (Conservative Guarantee):
  Evaluated across 4,000 posterior MCMC parameter trajectories.
  Certified Safe Operational Limit: T_safe = 58.7 deg C.
  
Warning: Operating at the point-estimate limit (64.4 deg C) exposes the fleet
to an unhedged 50% probability of exceeding the contractual 5% failure cap!
================================================================================
```



## Part 4: The 4 Mandatory Verification Gates (How to Audit the Agent)

The AI agent returned a clean, calibrated response: it centered the predictor,
distinguished the epistemic uncertainty of the mean line $\mu(x)$ from the
aleatoric predictive band $y_{\text{new}}$, and framed SLA provisioning around
the true predictive distribution.

However, as an **Epistemic Architect**, you must never accept an AI agent's
regression code or SLA estimates at face value. How do you know the agent didn't
secretly introduce collinearity, mistake the mean line for the individual
quantile guarantee, or produce divergent transitions on uncentered telemetry?

Before deploying the regression model to production autoscaling or contractual
SLA monitoring, you audit the agent against **four mandatory verification
gates**:

```
              THE 4 MANDATORY AGENTIC VERIFICATION GATES
┌────────────────────────────────────────────────────────────────────┐
│ Gate 1: Prior Predictive Check                                     │
│ Sane physical domain bounds before seeing real data                │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Domain Sanity Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 2: Simulation-Based Calibration (SBC)                         │
│ Algorithmic proof: recovers ground truth across 5,000 simulations  │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Algorithm Soundness Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 3: Mathematical Invariant Conservation                        │
│ Hierarchy shrink/pool invariants / PSIS Pareto-k < 0.7 / decay law │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Structural Invariants Verified]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 4: Posterior Predictive Retrodiction & Loss                   │
│ Sane predictive fit to telemetry & expected loss decision audit    │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │
                                  ▼ [SAFE FOR PRODUCTION DEPLOYMENT]
```

---

### Gate 1: Prior Predictive Check (Domain Sanity Gate)

* **The Production Risk (Why This Gate Exists)**:
  In automated infrastructure capacity planning, an uncalibrated AI agent will
  frequently assign naive "uninformative" priors such as
  $\alpha \sim \mathcal{N}(0, 100^2)$ and
  $\beta \sim \mathcal{N}(0, 10^2)$. In a latency regression across request
  volumes $x \in [100, 1000]$ QPS, such diffuse priors place massive
  probability mass on physically absurd realities: predicting negative latency
  ($y < 0\text{ ms}$) or astronomical latencies exceeding $10{,}000\text{ ms}$
  under normal operating conditions. In logistic GLMs for hardware failure,
  unscaled priors on log-odds induce complete separation ("Hauck-Donner effect")
  where prior failure probabilities collapse into binary spikes at $0$ or $1$.
  Skipping Gate 1 allows catastrophic prior misspecifications to warp posterior
  inference and corrupt downstream autoscaling policies.
* **First-Principles Mathematical Mechanism**:
  We evaluate the marginal distribution of synthetic observations
  $y_{\text{sim}}$ across the query domain by integrating over the prior
  joint distribution:
  $$p(y_{\text{sim}} \mid x) = \iiint \mathcal{N}(y_{\text{sim}} \mid
  \alpha_c + \beta(x - \bar{x}), \sigma) \, p(\alpha_c) \, p(\beta)
  \, p(\sigma) \, d\alpha_c \, d\beta \, d\sigma$$
  For our centered latency model with $\bar{x} = 550.0\text{ QPS}$:
  * Intercept prior: $\alpha_c \sim \mathcal{N}(120.0, 20.0^2)$.
  * Slope prior: $\beta \sim \mathcal{N}(0.08, 0.04^2)$.
  * Residual dispersion prior: $\sigma \sim \text{HalfNormal}(15.0)$ or
    $\text{Exponential}(\lambda = 0.067)$.
  At minimum load $x = 100\text{ QPS}$ ($x_c = -450$):
  $$\mathbb{E}[\mu(100)] = 120.0 + 0.08(-450) = 84.0\text{ ms}$$
  $$\text{Var}(\mu(100)) = 20.0^2 + (-450)^2(0.04)^2 = 400 + 324 = 724
  \implies \text{SD}_{\mu} \approx 26.91\text{ ms}$$
  Combining mean uncertainty with residual noise
  $\mathbb{E}[\sigma^2] = 15.0^2 = 225.0$:
  $$\text{SD}(y_{\text{sim}} \mid x=100) = \sqrt{724 + 225} \approx
  30.81\text{ ms}$$
  The prior probability of impossible negative latency is strictly bounded:
  $$P(y_{\text{sim}} < 0 \mid x=100) \approx
  \Phi\left(\frac{-84.0}{30.81}\right) = \Phi(-2.73) \approx 0.0032 <
  0.0050$$
  At maximum load $x = 1000\text{ QPS}$ ($x_c = +450$):
  $$\mathbb{E}[\mu(1000)] = 120.0 + 0.08(+450) = 156.0\text{ ms}, \quad
  P(y_{\text{sim}} > 1000\text{ ms}) < 10^{-6}$$
  For thermal logistic regression $z_i \sim \text{Bernoulli}(p_i)$ with
  $\text{logit}(p_i) = \gamma_c + \delta(T_i - \bar{T})$, priors
  $\gamma_c \sim \mathcal{N}(-3.0, 1.0^2)$ and
  $\delta \sim \mathcal{N}(0.15, 0.05^2)$ ensure prior crash probabilities
  span $[0.1\%, 50.0\%]$ smoothly without boundary collapse.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Non-Negativity Invariant: Across $10{,}000$ simulated prior trajectories,
    empirical $P(y_{\text{sim}} < 0) \le 0.0050$ (`[PASS]`; any prior
    generating $> 0.5\%$ negative latencies triggers immediate `[FAIL]`).
  * Upper Latency Bound: $P(y_{\text{sim}} > 1000\text{ ms}) \le 0.0010$
    (`[PASS]`).
  * Logistic Spread Invariant: Crash probabilities satisfy
    $P(0.001 \le p(T) \le 0.50) \ge 0.950$ across the operating range
    $T \in [40, 90]^\circ\text{C}$ (`[PASS]`).
  * Prior Slope Positivity: $P(\beta > 0) \ge 0.950$ (`[PASS]`; queuing theory
    dictates latency increases with load).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 1 (Prior Predictive Domain Sanity for Regression & GLM):
  Model: Latency y_i ~ Normal(alpha_c + beta*(x_i - 550), sigma).
  Priors: alpha_c ~ Normal(120, 20), beta ~ Normal(0.08, 0.04), sigma ~ HalfNormal(15).
  Execute 10,000 prior simulations across load grid x in [100, 1000] QPS:
  1. Draw alpha_c^(s), beta^(s), sigma^(s) from their respective priors.
  2. Simulate y_sim^(s)(x) = Normal(alpha_c^(s) + beta^(s)*(x - 550), sigma^(s)).
  Verify that:
  - Probability of negative latency P(y_sim < 0) <= 0.0050 at x = 100 QPS.
  - Probability of extreme latency P(y_sim > 1000 ms) <= 0.0010 at x = 1000 QPS.
  - Slope directionality P(beta > 0) >= 0.950.
  Report [PASS] with evaluated quantiles or [FAIL] with boundary violations.
  ```

---

### Gate 2: Simulation-Based Calibration (SBC) / Fake-Data Recovery

* **The Production Risk (Why This Gate Exists)**:
  When an AI assistant implements Bayesian regression code in PyMC, Stan, or
  NumPyro, subtle programming errors frequently creep in: flipped slope signs,
  misindexed predictor arrays, uncentered residual updates, or poorly configured
  NUTS sampler step sizes. In capacity planning, an underestimated slope
  parameter $\beta$ leads the autoscaler to provision insufficient container
  replicas during traffic surges, resulting in widespread request drops. Gate 2
  audits the mathematical engine against known ground truth across thousands of
  independent synthetic data realizations.
* **First-Principles Mathematical Mechanism**:
  SBC validates that the posterior sampler satisfies Bayes' joint
  self-consistency theorem:
  $$\iint p(\theta^*) \, p(y_{\text{fake}} \mid \theta^*) \,
  p(\theta \mid y_{\text{fake}}) \, d\theta^* \, dy_{\text{fake}}
  = p(\theta)$$
  The automated verification pipeline runs $M = 5{,}000$ Monte Carlo trials:
  1. Draw ground-truth parameter vectors
     $\theta^* = (\alpha_c^*, \beta^*, \sigma^*)$ from calibrated priors:
     $$\alpha_c^* \sim \mathcal{N}(120.0, 20.0^2), \quad
     \beta^* \sim \mathcal{N}(0.08, 0.04^2), \quad
     \sigma^* \sim \text{HalfNormal}(15.0)$$
  2. Generate synthetic observations $y_{\text{fake}, i} \sim
     \mathcal{N}(\alpha_c^* + \beta^*(x_i - \bar{x}), \sigma^*)$ for
     $N = 100$ loads $x_i \in [100, 1000]$.
  3. Condition the MCMC sampler on $y_{\text{fake}}$ to draw $S = 2{,}000$
     posterior samples.
  4. Compute equal-tailed $95\%$ credible intervals
     $[\theta_{0.025}, \theta_{0.975}]$ for each parameter.
  5. Check parameter inclusion: $h_m = \mathbb{I}(\theta^* \in
     [\theta_{0.025}, \theta_{0.975}])$.
  6. Compute the empirical rank statistic
     $U_m = \frac{1}{S}\sum_{s=1}^S \mathbb{I}(\theta^{(s)} < \theta^*)$.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Empirical Coverage Invariant: Across $M = 5{,}000$ independent trials,
    empirical coverage $\hat{C} = \frac{1}{M}\sum h_m$ must satisfy
    $|\hat{C} - 0.9500| \le 0.0080$ ($[94.20\%, 95.80\%]$) for all
    parameters $\alpha_c, \beta, \sigma$ (`[PASS]`).
  * Rank Uniformity Invariant: Kolmogorov-Smirnov test on rank statistics
    against $\text{Uniform}(0, 1)$ must yield $p_{\text{KS}} > 0.010$
    (`[PASS]`).
  * Recovery Bias Invariant: Mean recovery bias
    $|\frac{1}{M}\sum(\hat{\beta}_m - \beta^*)| \le 0.0010$ and
    $|\frac{1}{M}\sum(\hat{\sigma}_m - \sigma^*)| \le 0.150$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 2 (Simulation-Based Calibration on Bayesian Regression):
  Execute 5,000 Monte Carlo fake-data recovery trials:
  For trial m = 1 to 5,000:
    1. Draw ground truth alpha_c*, beta*, sigma* from priors.
    2. Generate synthetic dataset y_fake for N = 100 load points.
    3. Run NUTS MCMC (4 chains, 1000 warmup, 1000 draws).
    4. Compute 95% equal-tailed credible intervals for alpha_c, beta, sigma.
    5. Test inclusion of true parameters and compute rank statistics.
  Assert that empirical coverage is 95.0% ± 0.8% for all parameters.
  Assert KS test p-value > 0.01 for rank uniformity.
  Report [PASS] or [FAIL] with parameter recovery coverage table.
  ```

---

### Gate 3: Centering & Geometric Invariants (Orthogonality & MCMC Health)

* **The Production Risk (Why This Gate Exists)**:
  Fitting uncentered regression models where load $x \in [100, 1000]$
  ($\bar{x} = 550$) induces extreme mathematical collinearity between
  intercept $\alpha$ and slope $\beta$ ($\operatorname{Corr}(\alpha, \beta)
  \approx -0.98$). This collinearity forms an elongated, diagonally compressed
  posterior valley in parameter space. Hamiltonian Monte Carlo cannot traverse
  this geometry efficiently, triggering severe step-size shrinkage, divergent
  transitions, inflated $\hat{R} > 1.05$, and degraded effective sample sizes
  ($\text{ESS} < 100$). An uncalibrated AI agent will deliver sampling chains
  with hidden bias and unverified convergence. Gate 3 enforces predictor
  centering to ensure geometric orthogonality and flawless MCMC sampling.
* **First-Principles Mathematical Mechanism**:
  For an uncentered linear model $y_i = \alpha + \beta x_i + \epsilon_i$,
  the Fisher information matrix is:
  $$\mathcal{I}(\alpha, \beta) = \frac{1}{\sigma^2}\begin{pmatrix}
  N & \sum_{i=1}^N x_i \\
  \sum_{i=1}^N x_i & \sum_{i=1}^N x_i^2
  \end{pmatrix}$$
  Inverting $\mathcal{I}$ yields the parameter covariance:
  $$\operatorname{Cov}(\alpha, \beta) = -\frac{\bar{x}}{\sum_{i=1}^N (x_i
  - \bar{x})^2} \sigma^2 \implies \operatorname{Corr}(\alpha, \beta)
  = -\frac{\bar{x}}{\sqrt{\bar{x}^2 + \operatorname{Var}(x)}}$$
  With $\bar{x} = 550$ and $\operatorname{Var}(x) \approx 67{,}500$,
  $\operatorname{Corr}(\alpha, \beta) \approx -0.904$.
  By centering the predictor $x_{c, i} = x_i - \bar{x}$, we enforce
  $\sum_{i=1}^N x_{c, i} = 0$. The Fisher information matrix becomes strictly
  diagonal:
  $$\mathcal{I}(\alpha_c, \beta) = \frac{1}{\sigma^2}\begin{pmatrix}
  N & 0 \\
  0 & \sum_{i=1}^N x_{c, i}^2
  \end{pmatrix} \implies \operatorname{Cov}(\alpha_c, \beta) = 0$$
  The posterior geometry becomes orthogonal and isotropic. NUTS trajectories
  glide effortlessly along principal axes, eliminating energy divergences.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Orthogonality Invariant: Posterior correlation
    $|\operatorname{Corr}(\alpha_c, \beta)| \le 0.050$ strictly (`[PASS]`;
    any uncentered model yielding $|\operatorname{Corr}| > 0.80$ triggers
    `[FAIL]`).
  * Convergence Diagnostic: Gelman-Rubin $\hat{R} \le 1.005$ across all
    parameters (`[PASS]`; any $\hat{R} > 1.010$ triggers `[FAIL]`).
  * Zero Divergence Invariant: `divergent_transitions == 0` strictly across
    all chains (`[PASS]`; $\ge 1$ divergence triggers `[FAIL]`).
  * Sampling Efficiency: $\min(\text{ESS}_{\text{bulk}},
    \text{ESS}_{\text{tail}}) \ge 1{,}000$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 3 (Centering & MCMC Geometric Invariants):
  Extract posterior samples for alpha_c, beta, and sigma from NUTS trace:
  1. Compute empirical covariance matrix and parameter correlation:
     Corr(alpha_c, beta) = Cov(alpha_c, beta) / (SD(alpha_c) * SD(beta)).
  2. Verify |Corr(alpha_c, beta)| < 0.050.
  3. Extract diagnostic metrics: check max R_hat <= 1.005.
  4. Assert divergent transitions == 0 across all 4 chains.
  5. Check bulk and tail ESS >= 1,000 for all parameters.
  Report [PASS] with correlation and convergence summary or [FAIL].
  ```

---

### Gate 4: Posterior Predictive Retrodiction & Asymmetric SLA Loss

* **The Production Risk (Why This Gate Exists)**:
  Conflating the narrow epistemic credible band of the mean regression line
  $\mu(x)$ with the wide aleatoric posterior predictive band $y_{\text{new}}$.
  An uncalibrated AI agent will observe the $95\%$ credible interval of
  $\mu(x)$ at $800\text{ QPS}$ ($[136.5, 144.3]\text{ ms}$) and advise
  engineering leadership that a $145\text{ ms}$ contractual SLA is safe. In
  production reality, individual requests fluctuate around the mean line with
  residual standard deviation $\sigma \approx 15.0\text{ ms}$. Over $23\%$
  of incoming requests exceed $145\text{ ms}$, causing contractual breaches,
  financial SLA penalties, and emergency incident paging. Gate 4 guarantees
  that SLA decisions are derived from the true predictive distribution under
  asymmetric decision loss.
* **First-Principles Mathematical Mechanism**:
  We evaluate the exact decomposition of predictive variance at query load $x$:
  $$\text{Var}(\mu(x) \mid D) = \text{Var}(\alpha_c \mid D) + (x -
  \bar{x})^2 \text{Var}(\beta \mid D)$$
  $$\text{Var}(y_{\text{new}} \mid D) = \text{Var}(\mu(x) \mid D) +
  \mathbb{E}[\sigma^2 \mid D]$$
  At $x = 800\text{ QPS}$ ($x_c = +250$):
  * Epistemic uncertainty: $\text{Var}(\mu(800) \mid D) = 1.51^2 +
    250^2(0.005)^2 = 2.28 + 1.56 = 3.84 \implies \text{SD}_{\mu} \approx
    1.96\text{ ms}$.
    The 95% Credible Interval for the mean line is:
    $$[140.4 \pm 1.96(1.96)] = [136.5, 144.3]\text{ ms} \quad
    (\text{width } 7.8\text{ ms})$$
  * Aleatoric predictive variance: $\text{Var}(y_{\text{new}} \mid D) =
    3.84 + 14.88^2 = 3.84 + 221.41 = 225.25 \implies \text{SD}_{y} \approx
    15.01\text{ ms}$.
    The 95% Posterior Predictive Band for individual request latency is:
    $$[140.4 \pm 1.96(15.01)] = [110.8, 170.0]\text{ ms} \quad
    (\text{width } 59.2\text{ ms})$$
  Empirical retrodictive coverage check across observed telemetry:
  $$\hat{C}_{\text{retro}} = \frac{1}{N}\sum_{i=1}^N \mathbb{I}\left(y_i
  \in [y_{0.025}(x_i), y_{0.975}(x_i)]\right)$$
  Asymmetric SLA Decision Loss:
  Let $L(\text{SLA}, y) = C_{\text{breach}} \mathbb{I}(y > \text{SLA}) +
  C_{\text{cost}} \cdot \text{SLA}$. Sizing the contractual SLA to guarantee
  that at most $5\%$ of requests breach requires setting:
  $$\text{SLA}^* = F_{y_{\text{new}}}^{-1}(0.95 \mid x=800) = 170.0\text{ ms}$$
  Promising an SLA based on the mean line ($144.3\text{ ms}$) results in a
  catastrophic breach rate:
  $$P(y > 144.3 \mid x=800) \approx 1 - \Phi\left(\frac{144.3 -
  140.4}{15.01}\right) = 1 - \Phi(0.26) = 0.397 \quad (39.7\%)$$
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Retrodictive Coverage Invariant: Historical telemetry coverage
    $\hat{C}_{\text{retro}} \in [93.0\%, 97.0\%]$ (`[PASS]`; checking
    against $\mu(x)$ yields $\approx 10\%$, triggering immediate `[FAIL]`).
  * SLA Contractual Assertion: The certified $95\%$ SLA guarantee at $800$
    QPS must be $170.0 \pm 1.5\text{ ms}$ (`[PASS]`; proposing $\le
    145.0\text{ ms}$ triggers `[FAIL: CONFLATED_EPISTEMIC_ALEATORIC]`).
  * Certified Thermal Limit: $T_{\text{safe}} \le 58.7^\circ\text{C}$ at
    $95\%$ posterior confidence (`[PASS]`; proposing point-estimate
    $64.4^\circ\text{C}$ triggers `[FAIL]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 4 (Posterior Predictive Retrodiction & SLA Decision):
  Model: Latency regression on N = 100 historical points.
  1. Generate 4,000 posterior predictive simulations y_rep_i ~ Normal(mu(x_i), sigma).
  2. Compute 95% predictive interval for each data point and evaluate coverage.
  3. Assert empirical coverage is within 93.0% - 97.0%.
  4. Compute 95th percentile of y_new at load x = 800 QPS.
  5. Assert that contractual SLA recommendation equals 170.0 ± 1.5 ms.
  6. Flag [FAIL: CONFLATED_EPISTEMIC_ALEATORIC] if SLA recommendation is <= 145 ms.
  Report [PASS] or [FAIL] with evaluated SLA threshold.
  ```



## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Evaluation Dimension | Naive AI Assistant Output | Calibrated Probabilistic Specification |
| :--- | :--- | :--- |
| **Predictor Geometry** | Raw $x$: $\operatorname{Corr}(\alpha, \beta) \approx -0.98$, slow MCMC convergence | Centered $x - \bar{x}$: $\operatorname{Corr}(\alpha_c, \beta) \approx 0$, orthogonal Hessian |
| **Uncertainty Ribbon** | Plots mean line $\mu(x)$ ($\text{width} \approx 6\text{ms}$); ignores residual noise | Computes both $\mu(x)$ and $y_{\text{new}} \sim \mathcal{N}(\mu, \sigma)$ ($\text{width} \approx 59\text{ms}$) |
| **SLA Guarantee Risk** | Under-provisions SLA threshold by $25\text{ms}$; massive SLA breaches | Fully provisions for individual request dispersion with 95% coverage |
| **Binary Outcomes** | Linear regression; outputs negative failure rates ($p = -0.08$) | Logistic GLM; logit link bounds failure rates strictly in $[0, 1]$ |
| **Sampling Diagnostics** | Ignores divergences and ESS warnings | Enforces $\hat{R} < 1.01$, ESS $> 1000$, and zero divergences |
| **Decision Logic** | Symmetric point-estimate threshold ($p_{\text{mean}} < 0.05$) | Asymmetric 95th percentile upper confidence bound ($T_{\text{safe}}$) |


## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

Test your skills in auditing AI-generated probabilistic regression code.

---

### 🟢 Tier 1: Spot the AI Hallucination
An AI agent presents the following code to compute the "95% latency SLA limit":

```python
# Agent snippet
mu_post = trace.posterior["alpha"] + trace.posterior["beta"] * 900
sla_limit = float(np.percentile(mu_post, 95.0))
print(f"95% of all requests at 900 QPS will complete under {sla_limit:.1f} ms")
```

**Task**: Identify the critical bug in the agent's claim and explain why DevOps
will be paged within minutes of deploying this SLA.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
The agent computed the 95th percentile of the **posterior distribution of the
mean** $\mu(900)$, not the 95th percentile of the **posterior predictive
distribution** $y_{\text{new}}$.

The parameter $\mu(900)$ represents where the *average of an infinite number of
requests* sits. An individual user request suffers from aleatoric variance
$\sigma^2$. The code completely omitted drawing random normal perturbations
with standard deviation $\sigma$.

Because $\sigma \approx 15\text{ ms}$, approximately $50\%$ of requests at the
mean will exceed $\mu(900)$, and nearly $30\%$ of individual requests will
exceed the agent's `sla_limit`. The correct computation is:

```python
# Calibrated correction
mu_post = trace.posterior["alpha_c"] + trace.posterior["beta"] * (900 - x_bar)
sigma_post = trace.posterior["sigma"]
y_pred = np.random.normal(mu_post, sigma_post)
calibrated_sla = float(np.percentile(y_pred, 95.0))
```
</details>

---

### 🟡 Tier 2: The 3-Sentence Micro-Spec (Data, Prior, Loss/Action)
An engineer wants an agent to fit a multivariate latency regression model:
> *"Fit a linear model $y = \beta_0 + \beta_1 \cdot \text{mem} + \beta_2 \cdot
> \text{cache\_misses}$ to predict job latency, where memory is in GB
> ($[16, 512]$) and cache misses are in millions ($[1, 100]$)."*

**Task:** Write an everyday **3-sentence micro-spec** (Data, Prior/Model,
Loss/Action) that enforces mean centering, separates epistemic from aleatoric
uncertainty, and defines a capacity sizing decision rule. In your solution,
show the exact prompt, the mathematical calculation, and the resulting action
decision.

<details>
<summary><b>Click here to view the solution...</b></summary>

**1. Everyday 3-Sentence Micro-Spec:**
* **Data:** Ingest batch logs containing job completion time $y$ in seconds
  alongside centered covariates $m_c = \text{mem} - \bar{m}$ (GB) and $c_c =
  \text{cache\_misses} - \bar{c}$ (millions).
* **Prior/Model:** Fit $y \sim \mathcal{N}(\mu, \sigma)$ with $\mu = \alpha_c +
  \beta_m m_c + \beta_c c_c$, assigning weakly informative regularizing priors
  $\alpha_c \sim \mathcal{N}(60, 15)$, $\beta_m \sim \mathcal{N}(-0.1, 0.05)$,
  $\beta_c \sim \mathcal{N}(0.5, 0.2)$, and $\sigma \sim \text{Exp}(0.1)$.
* **Loss/Action:** Compute the 95th percentile of the posterior predictive
  distribution $\tilde{y}$ for high-load instances ($m=64\text{ GB}, c=50\text{
  M}$); allocate job timeout quota to $\tilde{y}_{0.95}$ if $\tilde{y}_{0.95} <
  120\text{ s}$, otherwise trigger container memory auto-scaling.

**2. Mathematical Calculation:**
Dataset means: $\bar{m} = 128\text{ GB}$, $\bar{c} = 20\text{ M}$. At target
workload ($m=64, c=50$):
$$m_c = 64 - 128 = -64\text{ GB}, \quad c_c = 50 - 20 = 30\text{ M}$$
Posterior parameter means:
$$\hat{\alpha}_c = 58.2, \quad \hat{\beta}_m = -0.09, \quad \hat{\beta}_c = 0.45, \quad \hat{\sigma} = 8.4\text{ s}$$
$$\mu_{\text{target}} = 58.2 + (-0.09 \times -64) + (0.45 \times 30) = 77.46\text{ s}$$
Aleatoric predictive 95th percentile:
$$\tilde{y}_{0.95} \approx \mu_{\text{target}} + 1.645 \cdot \hat{\sigma} = 77.46 + 1.645(8.4) = 91.28\text{ s}$$

**3. Resulting Action Decision:**
Because $\tilde{y}_{0.95} = 91.28\text{ s} < 120\text{ s}$, the cluster timeout
quota is set to $92\text{ s}$ without needing container auto-scaling.
</details>

---

### 🔴 Tier 3: Interrogate the Agent's Verification Gates
An autonomous agent generates a GLM to predict edge node crash risk from
temperature ($T$) and concurrent tasks ($K$). It claims the model is validated
because the overall predictive score is high ($R^2 = 0.88$). However, you
suspect severe **multicollinearity** between ambient temperature and CPU core
workload, inflating parameter uncertainty and rendering counterfactual
interventions invalid.

**Task:** Formulate a concise **2-line follow-up audit prompt** to challenge the
agent and expose whether multicollinearity corrupted Gate 1–4. Explain the exact
mathematical invariant and diagnostic test that catches it.

<details>
<summary><b>Click here to view the solution...</b></summary>

**2-Line Follow-Up Audit Prompt:**
```text
Compute the posterior correlation matrix among all regression slopes and report
Variance Inflation Factors (VIF) alongside coefficient marginal posteriors.
Contrast marginal slope HDI of temperature alone vs. conditioned on tasks
to prove identifiability without posterior correlation exceeding 0.70.
```

**What Invariant Catches It:**
* **Posterior Slope Correlation Invariant:** For identifiable, interpretable
  regression slopes:
  $$|\text{Corr}(\beta_i, \beta_j \mid \text{Data})| < 0.70, \quad \forall i \ne j$$
  If predictors are collinear, the posterior geometry forms an elongated
  ridge. The sampler explores a continuum where $\beta_1 + \beta_2 \approx
  \text{const}$, inflating $\text{Var}(\beta_1)$ and $\text{Var}(\beta_2)$
  even when the joint prediction appears stable.
* **Variance Inflation Factor (VIF) Gate:** In standard predictor screening,
  $\text{VIF}_j = 1 / (1 - R_j^2) < 5.0$. If $\text{VIF} \ge 5.0$, one
  predictor must be orthogonalized, regularized with a horseshoe prior, or
  dropped.
</details>

## Key Takeaways & Course Navigation

### Core Architectural Rules for AI Regression Systems
1. **Never Confuse $\mu(x)$ with $y_{\text{new}}$**: The mean line measures
   scientific epistemic knowledge; individual prediction bands capture the
   reality of production variation. Always size SLAs using posterior predictive
   draws.
2. **Always Center Continuous Predictors ($x - \bar{x}$)**: Centering decouples
   the intercept from slopes, eliminates collinear valleys, and restores
   efficient MCMC sampling.
3. **Use GLMs for Bounded Quantities**: Probabilities live on $[0, 1]$; rates
   live on $(0, \infty)$. Enforce logit or log link functions so the model
   respects physical reality.
4. **Demand the Four Verification Gates**: Before acting on AI recommendations,
   validate prior checks, fake-data recovery, parameter orthogonality, and
   posterior retrodiction coverage.

---

### 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](../README.md)**
* ⬅️ **Previous**: **[Ch 11: Two-Group t-Test (Freq vs. Bayes)](11_t_test_frequentist_vs_bayesian.ipynb)** *(Part II: Comparative Inference)*
* ➡️ **Next**: **[Ch 13: Hierarchical Models & Shrinkage](13_hierarchical_models_and_partial_pooling.ipynb)** *(Part III: Applied Modeling, Regression, Hierarchy & Model Comparison)*
* 🔄 **Companion Tracks for Chapter 12**: [📖 Conceptual](../conceptual/12_bayesian_regression_and_glms.ipynb) | [🤖 AI Track](12_bayesian_regression_and_glms.ipynb) | [🐍 Python](../python/12_bayesian_regression_and_glms.ipynb) | [🔵 R](../r/12_bayesian_regression_and_glms.ipynb)

